# **MLProcess - Smoke Detector**
---
**1 - Data Pipeline**

In [1]:
# Import the required libraries.
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

import src.utils as utils

## **0 - Data Definition**
---

| **Feature** | **Description** |
|---|---|
| `UTC` (datetime) | Time stamp the sensor records. |
| `Temperature[C]` (float) | Environmemt temperature (celcius). |
| `Humidity[%]` (float) | Percentage of environment temperature. |
| `TVOC[ppb]` (int) | Total Volatile Organic Compounts (parts per billion). |
| `eCO2[ppm]` (int) | CO2 concentration (parts per million). |
| `Raw H2` (int) | Number of H2 gas molecules. |
| `Raw Ethanol` (int) | Number of Ethanol gas molecules. |
| `Pressure[hPa]` (float) | Air pressure (hectopascal, 1hPa = 100Pa). |
| `PM1.0` (float) | Number of particles < 1.0 micrometer. |
| `PM2.5` (float) | Number of particles between 1.0 micrometer and 2.5 micrometer. |
| `NC0.5` (float) | Particle concentrate < 0.5 micrometer. |
| `NC1.0` (float) | Particle concentrate between 0.5 micrometer and 1.0 micrometer. |
| `NC2.5` (float) | Particle concentrate between 1.0 micrometer and 2.5 micrometer. |
| `CNT` (int) | Sample counter. |
---

**Output Variable (to be predicted):**

| **Feature** | **Description** |
|---|---|
| `Fire Alarm` (binary) | (0=no fire detected, 1=fire detected). |

## **1 - Configuration File**
---

In [2]:
# Load the conifguration file.
config = utils.load_config()
utils.print_debug("Config file loaded...")

2026-03-07 15:04:16.451158 Config file loaded...


In [3]:
# Check the configuration file.
config

{'columns_datetime': ['utc'],
 'columns_float': ['temperature',
  'humidity_pct',
  'pressure',
  'pm10',
  'pm25',
  'nc05',
  'nc10',
  'nc25'],
 'columns_int': ['tvoc', 'co2', 'raw_h2', 'raw_ethanol', 'fire_alarm'],
 'features': ['temperature',
  'humidity_pct',
  'pressure',
  'pm10',
  'tvoc',
  'co2',
  'raw_h2',
  'raw_ethanol'],
 'label': 'fire_alarm',
 'path_data_raw': 'data/raw/smoke_detection_iot.csv',
 'range_co2': [400, 60000],
 'range_fire_alarm': [0, 1],
 'range_humidity_pct': [0, 100],
 'range_nc05': [0, 65535],
 'range_nc10': [0, 65535],
 'range_nc25': [0, 65535],
 'range_pm10': [0, 65535],
 'range_pm25': [0, 65535],
 'range_pressure': [300, 1250],
 'range_raw_ethanol': [0, 60000],
 'range_raw_h2': [0, 60000],
 'range_temperature': [-40, 125],
 'range_tvoc': [0, 60000],
 'range_utc': ['8/6/2022', '14/6/2022']}

## **2 - Load Data**
---

In [3]:
# Function to load raw data.
def load_data(path_data):
    """
    Load csv files and return it as Pandas DataFrame.

    Parameters:
    ----------
    path_data : str
        Raw dataset location.

    Returns:
    -------
    raw_dataset : pd.DataFrame
        Loaded raw dataset.
    """

    # Load csv file.
    raw_dataset = pd.read_csv(path_data)

    # Check the original shape.
    print(f"Raw data shape       : {raw_dataset.shape}")

    # Drop any duplicate data.
    n_duplicates = raw_dataset.duplicated().sum()
    print(f"Number of duplicates : {n_duplicates}")

    raw_dataset = raw_dataset.drop_duplicates(keep="last")
    print(f"Dropped data shape   : {raw_dataset.shape}")

    return raw_dataset

In [4]:
# Load the raw dataset.
PATH_DATA_RAW = config["path_data_raw"]
raw_dataset = load_data(PATH_DATA_RAW)

Raw data shape       : (62630, 15)
Number of duplicates : 0
Dropped data shape   : (62630, 15)


In [5]:
# Check the raw dataset.
raw_dataset

,UTC,Temperature[C],Humidity[%],TVOC[ppb],eCO2[ppm],Raw H2,Raw Ethanol,Pressure[hPa],PM1.0,PM2.5,NC0.5,NC1.0,NC2.5,CNT,Fire Alarm
0,1654733331,20.000,57.36,0,400,12306,18520,939.735,0.00,0.00,0.00,0.000,0.000,0,0
1,1654733332,20.015,56.67,0,400,12345,18651,939.744,0.00,0.00,0.00,0.000,0.000,1,0
2,1654733333,20.029,55.96,0,400,12374,18764,939.738,0.00,0.00,0.00,0.000,0.000,2,0
3,1654733334,20.044,55.28,0,400,12390,18849,939.736,0.00,0.00,0.00,0.000,0.000,3,0
4,1654733335,20.059,54.69,0,400,12403,18921,939.744,0.00,0.00,0.00,0.000,0.000,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62625,1655130047,18.438,15.79,625,400,13723,20569,936.670,0.63,0.65,4.32,0.673,0.015,5739,0
62626,1655130048,18.653,15.87,612,400,13731,20588,936.678,0.61,0.63,4.18,0.652,0.015,5740,0
62627,1655130049,18.867,15.84,627,400,13725,20582,936.687,0.57,0.60,3.95,0.617,0.014,5741,0
62628,1655130050,19.083,16.04,638,400,13712,20566,936.680,0.57,0.59,3.92,0.611,0.014,5742,0


## **3 - Data Validation**
---

In [6]:
# Check the number of missing values.
raw_dataset.isnull().sum()

UTC               0
Temperature[C]    0
Humidity[%]       0
TVOC[ppb]         0
eCO2[ppm]         0
Raw H2            0
Raw Ethanol       0
Pressure[hPa]     0
PM1.0             0
PM2.5             0
NC0.5             0
NC1.0             0
NC2.5             0
CNT               0
Fire Alarm        0
dtype: int64

In [7]:
# Check the types of each features.
raw_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62630 entries, 0 to 62629
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   UTC             62630 non-null  int64  
 1   Temperature[C]  62630 non-null  float64
 2   Humidity[%]     62630 non-null  float64
 3   TVOC[ppb]       62630 non-null  int64  
 4   eCO2[ppm]       62630 non-null  int64  
 5   Raw H2          62630 non-null  int64  
 6   Raw Ethanol     62630 non-null  int64  
 7   Pressure[hPa]   62630 non-null  float64
 8   PM1.0           62630 non-null  float64
 9   PM2.5           62630 non-null  float64
 10  NC0.5           62630 non-null  float64
 11  NC1.0           62630 non-null  float64
 12  NC2.5           62630 non-null  float64
 13  CNT             62630 non-null  int64  
 14  Fire Alarm      62630 non-null  int64  
dtypes: float64(8), int64(7)
memory usage: 7.2 MB


- The `UTC` column has different type, it must be `datetime` type.

In [8]:
# Convert the UTC column to datetime type.
raw_dataset["UTC"] = pd.to_datetime(raw_dataset["UTC"], unit="s")

- Simplify the column names.
    - `Temperature[C]` -> `temperature`
    - `Humidity[%]` -> `humidity_pct`
    - `TVOC[PPB]` -> `tvoc`
    - `eCO2[ppm]` -> `co2`
    - `Raw H2` -> `raw_h2`
    - `Raw Ethanol` -> `raw_ethanol`
    - `Pressure[hPa]` -> `pressure`
    - `Fire Alarm` -> `fire_alarm`
    - lowercase the other column names

- Drop the `CNT` column.

In [9]:
# Drop the CNT column.
raw_dataset = raw_dataset.drop(columns=["CNT"])

In [10]:
# Rename the columns.
new_names = ["utc", "temperature", "humidity_pct", "tvoc", "co2", "raw_h2", "raw_ethanol", "pressure", "pm10", "pm25", "nc05", "nc10", "nc25", "fire_alarm"]

raw_dataset.columns = new_names

In [11]:
# Check the dataset.
raw_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62630 entries, 0 to 62629
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   utc           62630 non-null  datetime64[ns]
 1   temperature   62630 non-null  float64       
 2   humidity_pct  62630 non-null  float64       
 3   tvoc          62630 non-null  int64         
 4   co2           62630 non-null  int64         
 5   raw_h2        62630 non-null  int64         
 6   raw_ethanol   62630 non-null  int64         
 7   pressure      62630 non-null  float64       
 8   pm10          62630 non-null  float64       
 9   pm25          62630 non-null  float64       
 10  nc05          62630 non-null  float64       
 11  nc10          62630 non-null  float64       
 12  nc25          62630 non-null  float64       
 13  fire_alarm    62630 non-null  int64         
dtypes: datetime64[ns](1), float64(8), int64(5)
memory usage: 6.7 MB


In [12]:
# Serialize the validated data.
PATH_DATA_VALIDATED = "data/interim/validated_data.pkl"
utils.serialize_data(raw_dataset, PATH_DATA_VALIDATED)

Data serialized to data/interim/validated_data.pkl


['data/interim/validated_data.pkl']

In [13]:
# Update the configuration file.
config = utils.update_config(
    key = "path_data_validated",
    value = PATH_DATA_VALIDATED,
    config = config
)

Params Updated! 
Key: path_data_validated 
Value: data/interim/validated_data.pkl



## **4 - Update Range of Data**
---

In [14]:
# Update the range of features with minimum and maximum value of each column.
cols = config["features"]
cols_float = config["columns_float"]

print(f"Selected features : {cols}\n")

param_keys = [f"range_{col}" for col in cols]

for col, key in zip(cols, param_keys):
    if col in cols_float:
        min_value = float(np.min(raw_dataset[col]))
        max_value = float(np.max(raw_dataset[col]))        
    else:
        min_value = int(np.min(raw_dataset[col]))
        max_value = int(np.max(raw_dataset[col]))
    value = [min_value, max_value]
    
    config = utils.update_config(
        key = key,
        value = value,
        config = config
    )

Selected features : ['temperature', 'humidity_pct', 'pressure', 'pm10', 'tvoc', 'co2', 'raw_h2', 'raw_ethanol']

Params Updated! 
Key: range_temperature 
Value: [-22.01, 59.93]

Params Updated! 
Key: range_humidity_pct 
Value: [10.74, 75.2]

Params Updated! 
Key: range_pressure 
Value: [930.852, 939.861]

Params Updated! 
Key: range_pm10 
Value: [0.0, 14333.69]

Params Updated! 
Key: range_tvoc 
Value: [0, 60000]

Params Updated! 
Key: range_co2 
Value: [400, 60000]

Params Updated! 
Key: range_raw_h2 
Value: [10668, 13803]

Params Updated! 
Key: range_raw_ethanol 
Value: [15317, 21410]



## **5 - Data Defense**
---

- If `AssertionError` happens, there exists data that don't match the configuration.

In [15]:
# Function for data defense.
def data_defense(data, config):
    """
    Do data defense to check the data types and range.

    Parameters:
    ----------
    data : pd.DataFrame
        Validated data.

    config : dict
        Loaded configuration file.

    Returns:
    -------
    None, its a void function.
    """

    # Number of data.
    n_data = len(data)

    # List of columns.
    cols_float = config["columns_float"]
    cols_int = config["columns_int"]
    
    # Check data types.
    assert data.select_dtypes("float").columns.to_list() == cols_float, "an error occurs in float columns."
    assert data.select_dtypes("int").columns.to_list() == cols_int, "an error occurs in int columns."

    # Check range of data.
    for col in (cols_float + cols_int):
        min_value = config[f"range_{col}"][0]
        max_value = config[f"range_{col}"][1]
        assert data[col].between(min_value, max_value).sum() == n_data, f"an error occurs in {col} range."

In [16]:
# Do data defense.
data_defense(raw_dataset, config)

Seems like our data are in good condition!

## **6 - Data Split**
---

In [17]:
# Function for Input-Output Split.
def split_input_output(data, config):
    """
    Split the input(X) and output (y).

    Parameters:
    ----------
    data : pd.DataFrame
        The processed dataset.

    config : dict
        Loaded configuration parameters.

    Returns:
    -------
    X : pd.DataFrame
        The input data.

    y : pd.Series
        The output data.
    """

    # Ensure raw data immutable.
    data = data.copy()

    # Split the X and y.
    X = data[config["features"]]
    y = data[config["label"]]

    print(f"Original data shape : {data.shape}")
    print(f"Selected Features   : {config["features"]}")
    print(f"X data shape        : {X.shape}")
    print(f"y data shape        : {y.shape}")

    return X, y

In [18]:
# Split input-output.
X, y = split_input_output(
    data = raw_dataset,
    config = config
)

Original data shape : (62630, 14)
Selected Features   : ['temperature', 'humidity_pct', 'pressure', 'pm10', 'tvoc', 'co2', 'raw_h2', 'raw_ethanol']
X data shape        : (62630, 8)
y data shape        : (62630,)


In [19]:
# Function for Train-Test Split.
def split_train_test(X, y, test_size, random_state=123):
    """
    Split the train and test set.

    Parameters:
    ----------
    X : pd.DataFrame
        The input data.

    y : pd.Series
        The output data.

    test_size : float
        The proportion of test set.

    random_state : int, default = 123
        For reproducibility

    Returns:
    -------
    X_train, X_test : pd.DataFrame
        The train and test input.

    y_train, y_test : pd.Series
        The train and test output.
    """

    X_train, X_test, y_train, y_test = train_test_split(
                                            X, y,
                                            test_size = test_size,
                                            random_state = random_state,
                                            stratify = y
                                       )

    print(f"X_train shape : {X_train.shape}")
    print(f"y_train shape : {y_train.shape}")
    print(f"X_test shape  : {X_test.shape}")
    print(f"y_test shape  : {y_test.shape}\n")

    return X_train, X_test, y_train, y_test

In [20]:
# train:valid:test -> 80:10:10

# Train vs not-train.
X_train, X_not_train, y_train, y_not_train = split_train_test(
    X = X,
    y = y,
    test_size = 0.2
)

# Valid vs test.
X_valid, X_test, y_valid, y_test = split_train_test(
    X = X_not_train,
    y = y_not_train,
    test_size = 0.5,
)

X_train shape : (50104, 8)
y_train shape : (50104,)
X_test shape  : (12526, 8)
y_test shape  : (12526,)

X_train shape : (6263, 8)
y_train shape : (6263,)
X_test shape  : (6263, 8)
y_test shape  : (6263,)



In [21]:
# Sanity check the train data.
data_train = pd.concat(
    [X_train, y_train],
    axis = 1
)

data_train.sample(10)

,temperature,humidity_pct,pressure,pm10,tvoc,co2,raw_h2,raw_ethanol,fire_alarm
17439,15.897,51.89,938.754,1.44,1197,408,12893,19433,1
55367,41.490,35.72,936.859,0.44,9698,400,12877,19327,0
14752,14.116,53.15,938.853,2.09,1134,464,12860,19448,1
1679,24.968,48.46,939.608,1.17,47,400,13108,20003,0
54049,28.040,45.33,937.403,1.61,174,410,12782,20538,0
30681,19.680,54.79,939.704,2.21,53,400,13240,20198,1
49386,25.410,48.26,938.789,2.12,1425,452,12965,19368,1
34437,21.980,60.87,939.180,2.74,621,515,12885,19634,1
405,24.628,52.92,939.826,0.22,0,400,12725,19758,0
32971,18.300,56.61,939.410,0.23,308,400,13106,19961,1


## **7 - Data Serialization**
---

In [22]:
# Serialize the splitted data.
PATH_DATA_SPLITTED = "data/interim/"

data_configuration = {
    "train": {
        "X_train": X_train,
        "y_train": y_train
    },
    "valid": {
        "X_valid": X_valid,
        "y_valid": y_valid
    },
    "test": {
        "X_test": X_test,
        "y_test": y_test
    }
}

for key, value in data_configuration.items():
    config_key = f"path_data_{key}"
    config_value = []

    for v in value:
        # Get each path.
        path = f"{PATH_DATA_SPLITTED + v}.pkl"
        config_value.append(path)

        # Get each data.
        data = value[v]

        # Serialize the splitted data.
        utils.serialize_data(data, path)

    # Update the configuration parameters.
    config = utils.update_config(
        key = config_key,
        value = config_value,
        config = config        
    )

Data serialized to data/interim/X_train.pkl
Data serialized to data/interim/y_train.pkl
Params Updated! 
Key: path_data_train 
Value: ['data/interim/X_train.pkl', 'data/interim/y_train.pkl']

Data serialized to data/interim/X_valid.pkl
Data serialized to data/interim/y_valid.pkl
Params Updated! 
Key: path_data_valid 
Value: ['data/interim/X_valid.pkl', 'data/interim/y_valid.pkl']

Data serialized to data/interim/X_test.pkl
Data serialized to data/interim/y_test.pkl
Params Updated! 
Key: path_data_test 
Value: ['data/interim/X_test.pkl', 'data/interim/y_test.pkl']



In [24]:
# Check the configuration file.
config

{'columns_datetime': ['utc'],
 'columns_float': ['temperature',
  'humidity_pct',
  'pressure',
  'pm10',
  'pm25',
  'nc05',
  'nc10',
  'nc25'],
 'columns_int': ['tvoc', 'co2', 'raw_h2', 'raw_ethanol', 'fire_alarm'],
 'features': ['temperature',
  'humidity_pct',
  'pressure',
  'pm10',
  'tvoc',
  'co2',
  'raw_h2',
  'raw_ethanol'],
 'label': 'fire_alarm',
 'path_data_raw': 'data/raw/smoke_detection_iot.csv',
 'path_data_test': ['data/interim/X_test.pkl', 'data/interim/y_test.pkl'],
 'path_data_train': ['data/interim/X_train.pkl', 'data/interim/y_train.pkl'],
 'path_data_valid': ['data/interim/X_valid.pkl', 'data/interim/y_valid.pkl'],
 'path_data_validated': 'data/interim/validated_data.pkl',
 'range_co2': [400, 60000],
 'range_fire_alarm': [0, 1],
 'range_humidity_pct': [10.74, 75.2],
 'range_nc05': [0, 65535],
 'range_nc10': [0, 65535],
 'range_nc25': [0, 65535],
 'range_pm10': [0.0, 14333.69],
 'range_pm25': [0, 65535],
 'range_pressure': [930.852, 939.861],
 'range_raw_ethanol